In [0]:
#Ohio Motel Analysis 

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("HotelDataProcessing").getOrCreate()

spark

## 1. Load data

In [0]:
df = spark.table("workspace.default.ohio_house_motel_bookings_1")
display(df)

hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
Room 328,1,73,2016,October,44,24,1,2,2,0.0,0,BB,PRT,Online TA,TA/TO,0,0,0,D,D,0,No Deposit,8.0,null,0,Transient,113.7,0,1,Canceled,2016-08-16
Room 810,0,83,2016,May,19,1,2,3,2,0.0,0,BB,BEL,Direct,Direct,0,0,0,D,D,0,No Deposit,14.0,null,0,Transient,145.5,1,0,Check-Out,2016-05-06
Room 231,1,122,2017,January,4,28,2,5,2,1.0,0,BB,BRA,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,9.0,null,0,Transient,103.7,0,1,Canceled,2016-10-26
Room 193,1,175,2017,April,14,6,0,3,2,0.0,0,BB,IRL,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,null,0,Transient,60.0,0,0,Canceled,2017-02-22
Room 248,1,88,2015,July,29,18,2,2,1,0.0,0,BB,PRT,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,9.0,null,0,Transient,68.0,0,1,Canceled,2015-07-18
Room 903,0,1,2017,March,9,3,0,1,2,0.0,0,SC,ISR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,7.0,null,0,Transient,70.4,0,0,Check-Out,2017-03-04
Room 519,1,56,2017,March,12,21,0,3,2,0.0,0,BB,PRT,Offline TA/TO,TA/TO,0,0,0,A,A,0,Non Refund,86.0,null,0,Transient,85.0,0,0,Canceled,2017-01-24
Room 349,0,5,2016,April,18,27,0,1,1,0.0,0,BB,ESP,Direct,Direct,0,0,0,A,A,0,No Deposit,null,null,0,Transient-Party,121.0,0,0,Check-Out,2016-04-28
Room 136,0,210,2015,October,41,8,1,3,2,0.0,0,BB,FRA,Offline TA/TO,TA/TO,0,0,0,A,D,0,No Deposit,12.0,null,0,Transient-Party,72.75,0,0,Check-Out,2015-10-12
Room 680,0,1,2016,March,10,2,0,1,1,0.0,0,BB,PRT,Online TA,TA/TO,0,0,0,A,D,0,No Deposit,240.0,null,0,Transient,43.0,0,0,Check-Out,2016-03-03


In [0]:
%pip install pandas numpy seaborn matplotlib plotly

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 81.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [seaborn]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt





## 2. Profile the data
Check column types, non-null counts and row count before analyzing anything.

In [0]:
df.printSchema()

root
 |-- hotel: string (nullable = true)
 |-- is_canceled: long (nullable = true)
 |-- lead_time: long (nullable = true)
 |-- arrival_date_year: long (nullable = true)
 |-- arrival_date_month: string (nullable = true)
 |-- arrival_date_week_number: long (nullable = true)
 |-- arrival_date_day_of_month: long (nullable = true)
 |-- stays_in_weekend_nights: long (nullable = true)
 |-- stays_in_week_nights: long (nullable = true)
 |-- adults: long (nullable = true)
 |-- children: double (nullable = true)
 |-- babies: long (nullable = true)
 |-- meal: string (nullable = true)
 |-- country: string (nullable = true)
 |-- market_segment: string (nullable = true)
 |-- distribution_channel: string (nullable = true)
 |-- is_repeated_guest: long (nullable = true)
 |-- previous_cancellations: long (nullable = true)
 |-- previous_bookings_not_canceled: long (nullable = true)
 |-- reserved_room_type: string (nullable = true)
 |-- assigned_room_type: string (nullable = true)
 |-- booking_changes: lon

In [0]:
df.pandas_api().info()

Tried to attach usage logger `pyspark.databricks.pandas.usage_logger`, but an exception was raised: JVM wasn't initialised. Did you call it on executor side?


<class 'pyspark.pandas.frame.DataFrame'>
Index: 759482 entries, 0 to 759481
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           759482 non-null  object 
 1   is_canceled                     759482 non-null  int64  
 2   lead_time                       759482 non-null  int64  
 3   arrival_date_year               759482 non-null  int64  
 4   arrival_date_month              759482 non-null  object 
 5   arrival_date_week_number        759482 non-null  int64  
 6   arrival_date_day_of_month       759482 non-null  int64  
 7   stays_in_weekend_nights         759482 non-null  int64  
 8   stays_in_week_nights            759482 non-null  int64  
 9   adults                          759482 non-null  int64  
 10  children                        759443 non-null  float64
 11  babies                          759482 non-null  int64  
 12  meal              

In [0]:
df.columns

['hotel',
 'is_canceled',
 'lead_time',
 'arrival_date_year',
 'arrival_date_month',
 'arrival_date_week_number',
 'arrival_date_day_of_month',
 'stays_in_weekend_nights',
 'stays_in_week_nights',
 'adults',
 'children',
 'babies',
 'meal',
 'country',
 'market_segment',
 'distribution_channel',
 'is_repeated_guest',
 'previous_cancellations',
 'previous_bookings_not_canceled',
 'reserved_room_type',
 'assigned_room_type',
 'booking_changes',
 'deposit_type',
 'agent',
 'company',
 'days_in_waiting_list',
 'customer_type',
 'adr',
 'required_car_parking_spaces',
 'total_of_special_requests',
 'reservation_status',
 'reservation_status_date']

## 3. Clean the data

| Issue | Decision | Why |
|---|---|---|
| Negative ADR or a booking with 0 guests | Remove | Physically impossible, so these are data errors |
| Exact duplicate rows | Remove | `hotel` holds the room number, so two identical rows mean the same room, same dates and same guest details, almost certainly a double-loaded record |
| `company` (mostly empty) and `agent` (an ID with no meaning on its own) | Drop the columns | Not useful for the questions asked here |
| `country` missing | Fill with `"Unknown"` | Keeps the rows; "Unknown" becomes its own category |
| `children` missing (39 rows) | Fill with 0 | Tiny share of rows, and 0 is the most common value |

Section 6 re-runs this whole pipeline and logs how many rows each step removed.

In [0]:
# Impossible values: negative price, or a booking with no guests.
# coalesce() treats a missing children value as 0. Without it, adults + null + babies = null,
# and filter(~bad) silently drops every row where children is null (see section 6.1).
bad = (F.col("adr") < 0) | ((F.col("adults") + F.coalesce(F.col("children"), F.lit(0)) + F.col("babies")) == 0)
print("Impossible rows removed:", df.filter(bad).count())
df = df.filter(~bad)

Impossible rows removed: 1203


In [0]:
# Remove exact duplicate rows (same room, dates, guests, price and status)
before = df.count()
df = df.dropDuplicates()
print(f"Duplicate rows removed: {before - df.count():,}")

Duplicate rows removed: 18,095


In [0]:
df.pandas_api().isna().sum()

/databricks/python/lib/python3.12/site-packages/pyspark/pandas/generic.py:1372: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  warnings.warn(


hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                              39
babies                                 0
meal                                   0
country                             2956
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                             100586
company         

In [0]:
df = (df.drop("agent", "company")
              .withColumn("children", F.col("children").cast("int"))
              .fillna({"country": "Unknown", "children": 0}))

df.pandas_api().isna().sum()

/databricks/python/lib/python3.12/site-packages/pyspark/pandas/generic.py:1372: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  warnings.warn(


hotel                             0
is_canceled                       0
lead_time                         0
arrival_date_year                 0
arrival_date_month                0
arrival_date_week_number          0
arrival_date_day_of_month         0
stays_in_weekend_nights           0
stays_in_week_nights              0
adults                            0
children                          0
babies                            0
meal                              0
country                           0
market_segment                    0
distribution_channel              0
is_repeated_guest                 0
previous_cancellations            0
previous_bookings_not_canceled    0
reserved_room_type                0
assigned_room_type                0
booking_changes                   0
deposit_type                      0
days_in_waiting_list              0
customer_type                     0
adr                               0
required_car_parking_spaces       0
total_of_special_requests   

In [0]:
df.pandas_api().info()

<class 'pyspark.pandas.frame.DataFrame'>
Index: 740184 entries, 0 to 740183
Data columns (total 30 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           740184 non-null  object 
 1   is_canceled                     740184 non-null  int64  
 2   lead_time                       740184 non-null  int64  
 3   arrival_date_year               740184 non-null  int64  
 4   arrival_date_month              740184 non-null  object 
 5   arrival_date_week_number        740184 non-null  int64  
 6   arrival_date_day_of_month       740184 non-null  int64  
 7   stays_in_weekend_nights         740184 non-null  int64  
 8   stays_in_week_nights            740184 non-null  int64  
 9   adults                          740184 non-null  int64  
 10  children                        740184 non-null  int32  
 11  babies                          740184 non-null  int64  
 12  meal              

In [0]:
df.select('lead_time').show(5)

+---------+
|lead_time|
+---------+
|        0|
|       55|
|      113|
|        2|
|        0|
+---------+
only showing top 5 rows


## 4. Outliers: do they distort the results?
IQR flags far too many rows on columns that are mostly 0 or small whole numbers (`adults`, `children`, `booking_changes`), so IQR counts alone aren't a reason to delete anything. The real test is whether capping the extremes at the 1st and 99th percentiles changes the averages.

In [0]:
pdf = df.toPandas().select_dtypes("number")
q1, q3 = pdf.quantile(0.25), pdf.quantile(0.75)
iqr = q3 - q1
((pdf < q1 - 1.5 * iqr) | (pdf > q3 + 1.5 * iqr)).sum()

is_canceled                            0
lead_time                          17938
arrival_date_year                      0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights             1588
stays_in_week_nights               21321
adults                            183498
children                           54617
babies                              5829
is_repeated_guest                  23399
previous_cancellations             37964
previous_bookings_not_canceled     22906
booking_changes                   113065
days_in_waiting_list               22092
adr                                24256
required_car_parking_spaces        47399
total_of_special_requests          17881
dtype: int64

In [0]:
display(df.summary())

summary,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status
count,740184,740184,740184,740184,740184,740184,740184,740184,740184,740184,740184,740184,740184,740184,740184,740184,740184,740184,740184,740184,740184,740184,740184,740184,740184,740184,740184,740184,740184
mean,null,0.36287733860769755,102.38810079655869,2016.1575135371745,null,27.148892707759153,15.793766144634308,0.9344946661911092,2.506914496935897,1.861557396539239,0.10660727602866314,0.008239843066048441,null,null,null,null,0.03161240988727127,0.08351301838461787,0.1442708839964117,null,null,0.2220083114468835,null,2.2541354582103907,null,102.04727374544504,0.06438128897679496,0.578280535650595,null
stddev,null,0.4808302071254824,105.57676475085758,0.7048708257806244,null,13.591407257160094,8.78582052561861,0.9948209121885594,1.9036485033544284,0.5962600895177838,0.4032351786963836,0.10210512163026579,null,null,null,null,0.17496601609218243,0.8299890730214716,1.5765005401706789,null,null,0.6436081104282775,null,17.421479844700503,null,50.47448337656058,0.2477377171728156,0.7933499689902204,null
min,Room 101,0,0,2015,April,1,1,0,0,0,0,0,BB,ABW,Aviation,Corporate,0,0,0,A,A,0,No Deposit,0,Contract,0.0,0,0,Canceled
25%,null,0,18,2016,null,16,8,0,1,2,0,0,null,null,null,null,0,0,0,null,null,0,null,0,null,69.56,0,0,null
50%,null,0,68,2016,null,27,16,1,2,2,0,0,null,null,null,null,0,0,0,null,null,0,null,0,null,95.0,0,0,null
75%,null,1,158,2017,null,38,23,2,3,2,0,0,null,null,null,null,0,0,0,null,null,0,null,0,null,126.0,0,1,null
max,Room 999,1,737,2017,September,53,31,19,50,55,10,10,Undefined,ZWE,Undefined,Undefined,1,26,72,L,L,18,Refundable,391,Transient-Party,5400.0,8,5,No-Show


In [0]:
for c in ["adr", "lead_time", "stays_in_week_nights"]:
    lo, hi = pdf[c].quantile([0.01, 0.99])
    print(c, "mean:", round(pdf[c].mean(), 2),
          "| capped mean:", round(pdf[c].clip(lo, hi).mean(), 2),
          "| median:", round(pdf[c].median(), 2))

adr mean: 102.05 | capped mean: 101.68 | median: 95.0
lead_time mean: 102.39 | capped mean: 101.8 | median: 68.0
stays_in_week_nights mean: 2.51 | capped mean: 2.49 | median: 2.0


**Outlier conclusion**

| Column | Mean | Capped mean | Change | Decision |
|---|---|---|---|---|
| adr | 102.05 | 101.68 | -0.4% | Keep outliers |
| lead_time | 102.39 | 101.80 | -0.6% | Keep outliers, report the **median** (68 days) |
| stays_in_week_nights | 2.51 | 2.49 | -0.8% | Keep outliers |

Capping moved every average by less than 1%, so outliers are not distorting the results. Lead time has a mean of 102 days and a median of 68: the distribution is skewed (many early bookers), not driven by a few extreme rows, so the median is the honest "typical" value.

## 5. Where guests come from
Guests = adults + children + babies, counted only for bookings that were not canceled.

In [0]:
import plotly.express as px

guests = (df.filter(F.col("is_canceled") == 0)   # only bookings that actually arrived
          .withColumn("guests", F.col("adults") + F.col("children") + F.col("babies"))
          .groupBy("country").agg(F.sum("guests").alias("guests"))
          .orderBy(F.desc("guests"))
          .toPandas())

In [0]:
fig = px.choropleth(guests, locations="country", color="guests",
                    hover_name="country", color_continuous_scale="Blues",
                    title="Number of Guests by Country")
fig.show()

<!doctype html>

In [0]:
fig2 = px.bar(guests.head(10), x="country", y="guests", text="guests",
              title="Top 10 Countries by Number of Guests")
fig2.show()

<!doctype html>

## 6. Data checks during analysis
Profiling caught the obvious problems up front. The checks below look for problems that only show up once the analysis is underway: a bug in my own cleaning logic, rules every valid booking must follow, columns that must agree with each other, and gaps in time coverage. Every check prints a count so its impact can be measured.

### 6.1 A bug in my own cleaning filter
The first version of the impossible-values filter was `adults + children + babies == 0`. At that point `children` still had nulls, and in Spark `null + anything = null`. The condition was therefore null for those rows, and `filter(~bad)` drops rows where the condition is null. The 39 rows with missing `children` were being deleted silently instead of filled with 0. Section 3 now uses `coalesce()` to fix it.

In [0]:
RAW_TABLE = "workspace.default.ohio_house_motel_bookings_1"
raw = spark.table(RAW_TABLE)

old_bad = (F.col("adr") < 0) | ((F.col("adults") + F.col("children") + F.col("babies")) == 0)
print("Rows the old filter silently dropped (condition was null):", raw.filter(old_bad.isNull()).count())

Rows the old filter silently dropped (condition was null): 39


### 6.2 Rule checks: what must be true for every booking

In [0]:
nights = F.col("stays_in_weekend_nights") + F.col("stays_in_week_nights")
guests = F.col("adults") + F.col("children") + F.col("babies")

rules = {
    "zero-night stay":                          nights == 0,
    "canceled but status is Check-Out":         (F.col("is_canceled") == 1) & (F.col("reservation_status") == "Check-Out"),
    "not canceled but status Canceled/No-Show": (F.col("is_canceled") == 0) & (F.col("reservation_status") != "Check-Out"),
    "children/babies with no adults":           (F.col("adults") == 0) & (guests > 0),
    "ADR = 0 and not Complementary":            (F.col("adr") == 0) & (F.col("market_segment") != "Complementary"),
}

total = df.count()
counts = df.agg(*[F.sum(cond.cast("int")).alias(name) for name, cond in rules.items()]).first()
rule_checks = pd.DataFrame({"rule": list(rules), "rows": [counts[r] or 0 for r in rules]})
rule_checks["pct_of_rows"] = (100 * rule_checks["rows"] / total).round(2)
rule_checks

,rule,rows,pct_of_rows
0,zero-night stay,3912,0.53
1,canceled but status is Check-Out,0,0.00
2,not canceled but status Canceled/No-Show,0,0.00
3,children/babies with no adults,1503,0.20
4,ADR = 0 and not Complementary,7122,0.96


**Decisions**
- **Zero-night stays:** remove. No night was sold, so there is no room revenue, and their ADR (often 0) drags price averages down.
- **Status mismatches:** remove. `is_canceled` is the main thing being analyzed, so rows where it contradicts the reservation status can't be trusted.
- **Children with no adults, and ADR = 0 outside Complementary:** keep. Both can happen in real life (a family booking under a child's name, staff or promotional stays), and they are a small share of rows.

In [0]:
status_mismatch = rules["canceled but status is Check-Out"] | rules["not canceled but status Canceled/No-Show"]

before = df.count()
df = df.filter((nights > 0) & ~status_mismatch)
print(f"Removed {before - df.count():,} rows ({100 * (before - df.count()) / before:.2f}%) | rows left: {df.count():,}")

Removed 3,912 rows (0.53%) | rows left: 736,272


### 6.3 Time coverage: are all years complete?

In [0]:
coverage = (df.groupBy("arrival_date_year")
              .agg(F.countDistinct("arrival_date_month").alias("months_of_data"),
                   F.count("*").alias("bookings"))
              .orderBy("arrival_date_year"))
display(coverage)

arrival_date_year,months_of_data,bookings
2015,6,133794
2016,12,351668
2017,8,250810


**Finding:** 2015 only has data from July and 2017 stops in August, so only 2016 is a full year. Comparing yearly totals would make 2016 look like a boom year and 2017 look like a decline, when those years just have fewer months of data.

**Fix:** in the seasonality analysis (8.2), monthly figures are divided by the number of years of data for that month, so July and August (3 years of data) are compared fairly against other months (2 years).

### 6.4 Cleaning log: how many rows each step removed
The function below re-runs every cleaning step from the raw table, so the whole pipeline is in one place and each step's impact is recorded.

In [0]:
def clean_pipeline(raw, dedup=True, fill_nulls=True, track=True):
    """Run every cleaning step from the raw table. Returns (clean_df, [(step, rows_after), ...])."""
    steps = []
    log = lambda name, d: steps.append((name, d.count())) if track else None

    d = raw
    log("raw data", d)

    no_guests = (F.col("adults") + F.coalesce(F.col("children"), F.lit(0)) + F.col("babies")) == 0
    d = d.filter(~((F.col("adr") < 0) | no_guests))
    log("remove negative ADR / zero guests", d)

    if dedup:
        d = d.dropDuplicates()
        log("remove duplicate rows", d)

    d = d.drop("agent", "company").withColumn("children", F.col("children").cast("int"))
    if fill_nulls:
        d = d.fillna({"country": "Unknown", "children": 0})
    else:
        d = d.dropna(subset=["country", "children"])
        log("drop rows with null country/children", d)

    n = F.col("stays_in_weekend_nights") + F.col("stays_in_week_nights")
    status_ok = (((F.col("is_canceled") == 1) & (F.col("reservation_status") != "Check-Out")) |
                 ((F.col("is_canceled") == 0) & (F.col("reservation_status") == "Check-Out")))
    d = d.filter((n > 0) & status_ok)
    log("remove zero-night stays / status mismatches", d)
    return d, steps


_, steps = clean_pipeline(raw)
cleaning_log = pd.DataFrame(steps, columns=["step", "rows_after"])
cleaning_log["rows_removed"] = (-cleaning_log["rows_after"].diff()).fillna(0).astype(int)
cleaning_log["pct_of_raw"] = (100 * cleaning_log["rows_removed"] / cleaning_log.loc[0, "rows_after"]).round(2)
print("Pipeline result matches the notebook's df:", cleaning_log["rows_after"].iloc[-1] == df.count())
cleaning_log

Pipeline result matches the notebook's df: True


,step,rows_after,rows_removed,pct_of_raw
0,raw data,759482,0,0.00
1,remove negative ADR / zero guests,758279,1203,0.16
2,remove duplicate rows,740184,18095,2.38
3,remove zero-night stays / status mismatches,736272,3912,0.52


### 6.5 Sensitivity check: do the cleaning choices change the answer?
The same key metrics are calculated under different cleaning choices. If they barely move, the conclusions don't depend on those choices. If they move a lot, that needs to be reported.

In [0]:
def key_metrics(d):
    r = d.agg(F.count("*").alias("rows"),
              F.avg("is_canceled").alias("cancel_rate"),
              F.avg("adr").alias("avg_adr")).first()
    return r["rows"], round(100 * r["cancel_rate"], 2), round(r["avg_adr"], 2)

variants = {
    "final (dedup + fill nulls)":     clean_pipeline(raw, track=False)[0],
    "keep duplicates":                clean_pipeline(raw, dedup=False, track=False)[0],
    "drop null rows instead of fill": clean_pipeline(raw, fill_nulls=False, track=False)[0],
}
sensitivity = pd.DataFrame([(name, *key_metrics(d)) for name, d in variants.items()],
                           columns=["cleaning choice", "rows", "cancel_rate_pct", "avg_adr"])
base = sensitivity.iloc[0]
sensitivity["cancel_rate_change_pts"] = (sensitivity["cancel_rate_pct"] - base["cancel_rate_pct"]).round(2)
sensitivity["avg_adr_change_pct"] = (100 * (sensitivity["avg_adr"] / base["avg_adr"] - 1)).round(2)
sensitivity

,cleaning choice,rows,cancel_rate_pct,avg_adr,cancel_rate_change_pts,avg_adr_change_pct
0,final (dedup + fill nulls),736272,36.46,102.59,0.00,0.00
1,keep duplicates,754339,37.32,102.41,0.86,-0.18
2,drop null rows instead of fill,733314,36.55,102.76,0.09,0.17


## 7. Feature engineering
New columns used in the revenue analysis:
- `month_num`, `arrival_date`: real dates, so months sort in calendar order
- `total_nights`, `total_guests`
- `revenue` = ADR × nights, the estimated room revenue of the booking (meals and extras are not in the data)
- `realized_revenue`: revenue from bookings that were not canceled
- `lost_revenue`: revenue from bookings that were canceled
- `room_changed`: the guest got a different room type than they reserved
- `lead_time_bucket`: how far ahead the booking was made

Note: in this version of the dataset the `hotel` column holds room numbers ("Room 328"), not hotel type, so it is treated as a room ID.

In [0]:
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

MONTHS = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]
LEAD_ORDER = ["0-7 days", "8-30 days", "31-90 days", "91-180 days", "181-365 days", "365+ days"]
month_map = F.create_map(*[x for i, m in enumerate(MONTHS, 1) for x in (F.lit(m), F.lit(i))])

df = (df
      .withColumn("month_num", month_map[F.col("arrival_date_month")])
      .withColumn("arrival_date", F.expr("make_date(arrival_date_year, month_num, arrival_date_day_of_month)"))
      .withColumn("total_nights", F.col("stays_in_weekend_nights") + F.col("stays_in_week_nights"))
      .withColumn("total_guests", F.col("adults") + F.col("children") + F.col("babies"))
      .withColumn("revenue", F.round(F.col("adr") * F.col("total_nights"), 2))
      .withColumn("realized_revenue", F.when(F.col("is_canceled") == 0, F.col("revenue")).otherwise(F.lit(0.0)))
      .withColumn("lost_revenue", F.when(F.col("is_canceled") == 1, F.col("revenue")).otherwise(F.lit(0.0)))
      .withColumn("room_changed", (F.col("reserved_room_type") != F.col("assigned_room_type")).cast("int"))
      .withColumn("lead_time_bucket",
                  F.when(F.col("lead_time") <= 7, LEAD_ORDER[0])
                   .when(F.col("lead_time") <= 30, LEAD_ORDER[1])
                   .when(F.col("lead_time") <= 90, LEAD_ORDER[2])
                   .when(F.col("lead_time") <= 180, LEAD_ORDER[3])
                   .when(F.col("lead_time") <= 365, LEAD_ORDER[4])
                   .otherwise(LEAD_ORDER[5])))

print(df.agg(F.min("arrival_date").alias("first"), F.max("arrival_date").alias("last")).first())
display(df.select("arrival_date", "total_nights", "adr", "revenue", "realized_revenue",
                  "lost_revenue", "room_changed", "lead_time_bucket").limit(10))

Row(first=datetime.date(2015, 7, 1), last=datetime.date(2017, 8, 31))


arrival_date,total_nights,adr,revenue,realized_revenue,lost_revenue,room_changed,lead_time_bucket
2015-08-13,2,85.0,170.0,170.0,0.0,1,0-7 days
2015-12-30,4,146.5,586.0,0.0,586.0,0,31-90 days
2016-07-08,7,190.0,1330.0,0.0,1330.0,0,91-180 days
2017-02-16,1,35.0,35.0,35.0,0.0,1,0-7 days
2016-08-02,1,120.7,120.7,120.7,0.0,0,0-7 days
2016-07-11,4,81.26,325.04,325.04,0.0,0,91-180 days
2016-10-27,2,90.0,180.0,180.0,0.0,0,0-7 days
2015-09-21,1,67.0,67.0,67.0,0.0,1,181-365 days
2017-04-24,3,99.0,297.0,297.0,0.0,0,91-180 days
2017-08-08,6,89.1,534.6,534.6,0.0,0,181-365 days


## 8. Revenue analysis
`seg_summary()` gives the same scorecard for any column, so each question below is one call:
bookings, cancellation rate, average ADR, realized revenue, lost revenue, and **revenue per booking made** (realized revenue ÷ all bookings, including canceled ones). That last metric shows what a booking from a segment is actually worth once its cancellations are taken into account.

In [0]:
def seg_summary(data, group_col, min_bookings=0):
    return (data.groupBy(group_col)
                .agg(F.count("*").alias("bookings"),
                     F.round(100 * F.avg("is_canceled"), 1).alias("cancel_rate_pct"),
                     F.round(F.avg("adr"), 2).alias("avg_adr"),
                     F.round(F.sum("realized_revenue"), 0).alias("realized_revenue"),
                     F.round(F.sum("lost_revenue"), 0).alias("lost_revenue"))
                .withColumn("revenue_per_booking", F.round(F.col("realized_revenue") / F.col("bookings"), 2))
                .filter(F.col("bookings") >= min_bookings))

### 8.1 Cancellations and lost revenue
Which bookings cancel, and how much revenue do cancellations cost?

In [0]:
kpi = df.agg(F.count("*").alias("bookings"),
             F.avg("is_canceled").alias("cancel_rate"),
             F.sum("realized_revenue").alias("realized"),
             F.sum("lost_revenue").alias("lost")).first()

print(f"Bookings:                    {kpi['bookings']:,}")
print(f"Cancellation rate:           {100 * kpi['cancel_rate']:.1f}%")
print(f"Realized room revenue:       ${kpi['realized']:,.0f}")
print(f"Revenue lost to cancels:     ${kpi['lost']:,.0f} "
      f"({100 * kpi['lost'] / (kpi['realized'] + kpi['lost']):.1f}% of all booked revenue)")

Bookings:                    736,272
Cancellation rate:           36.5%
Realized room revenue:       $162,958,621
Revenue lost to cancels:     $103,237,503 (38.8% of all booked revenue)


In [0]:
segment_df = seg_summary(df, "market_segment").orderBy(F.desc("lost_revenue")).toPandas()
display(segment_df)

fig = px.bar(segment_df, x="market_segment", y="lost_revenue", text_auto=".3s",
             hover_data=["bookings", "cancel_rate_pct"],
             title="Revenue lost to cancellations by market segment")
fig.update_layout(xaxis_title=None, yaxis_title="Lost revenue ($)")
fig.show()

market_segment,bookings,cancel_rate_pct,avg_adr,realized_revenue,lost_revenue,revenue_per_booking
Online TA,354953,37.1,117.83,8.6308427E7,6.5015169E7,243.15
Groups,116301,59.4,79.28,1.151261E7,1.5993232E7,98.99
Offline TA/TO,146993,33.1,87.43,3.5462892E7,1.4746493E7,241.26
Direct,78969,15.5,116.28,2.5583872E7,6223084.0,323.97
Corporate,32883,18.0,69.57,3624351.0,1167934.0,110.22
Aviation,1502,19.3,101.17,436429.0,89781.0,290.57
Complementary,4650,13.6,3.05,30039.0,1126.0,6.46
Undefined,21,100.0,17.14,0.0,684.0,0.0


<!doctype html>

In [0]:
lead_df = seg_summary(df, "lead_time_bucket").toPandas()
lead_df["lead_time_bucket"] = pd.Categorical(lead_df["lead_time_bucket"], LEAD_ORDER, ordered=True)
lead_df = lead_df.sort_values("lead_time_bucket")
display(lead_df)

fig = px.bar(lead_df, x="lead_time_bucket", y="cancel_rate_pct", text="cancel_rate_pct",
             title="Cancellation rate by how far ahead the booking was made")
fig.update_layout(xaxis_title="Lead time", yaxis_title="Cancellation rate (%)")
fig.show()

lead_time_bucket,bookings,cancel_rate_pct,avg_adr,realized_revenue,lost_revenue,revenue_per_booking
0-7 days,121459,9.8,91.95,2.0585421E7,2828794.0,169.48
8-30 days,119097,27.6,108.65,2.7502602E7,1.276872E7,230.93
31-90 days,182488,37.0,107.23,4.3844779E7,2.670527E7,240.26
91-180 days,164194,44.1,109.32,4.2578856E7,3.177089E7,259.32
181-365 days,130737,54.6,95.34,2.6699412E7,2.712101E7,204.22
365+ days,18297,67.4,78.9,1747552.0,2042819.0,95.51


<!doctype html>

In [0]:
# Other cancellation drivers
deposit_df = seg_summary(df, "deposit_type").orderBy(F.desc("cancel_rate_pct")).toPandas()
display(deposit_df)

display(seg_summary(df, "customer_type").orderBy(F.desc("cancel_rate_pct")))

display(seg_summary(df.withColumn("has_cancelled_before", F.col("previous_cancellations") > 0),
                    "has_cancelled_before"))

deposit_type,bookings,cancel_rate_pct,avg_adr,realized_revenue,lost_revenue,revenue_per_booking
Non Refund,82449,99.2,89.37,116443.0,2.029483E7,1.41
No Deposit,652724,28.6,104.3,1.62648731E8,8.2798785E7,249.18
Refundable,1099,21.8,78.37,193446.0,143888.0,176.02


customer_type,bookings,cancel_rate_pct,avg_adr,realized_revenue,lost_revenue,revenue_per_booking
Transient,553784,40.2,107.83,1.21419836E8,9.201921E7,219.25
Contract,25160,29.7,88.59,9485780.0,2089997.0,377.02
Transient-Party,153698,24.7,86.49,3.1346644E7,8993156.0,203.95
Group,3630,10.4,81.82,706360.0,135140.0,194.59


has_cancelled_before,bookings,cancel_rate_pct,avg_adr,realized_revenue,lost_revenue,revenue_per_booking
true,37952,90.9,76.21,426322.0,7811465.0,11.23
false,698320,33.5,104.02,1.62532299E8,9.5426038E7,232.75


**Watch out with deposit type:** if "Non Refund" bookings show a very high cancellation rate, that "lost" revenue was mostly **kept** by the hotel, since the deposit isn't returned. The lost-revenue figure is therefore an upper bound. The real loss is concentrated in "No Deposit" bookings.

### 8.2 Seasonality: when is demand high, and is price keeping up?
Monthly figures are divided by the number of years of data for each month (the partial-year issue from 6.3).

In [0]:
monthly_df = (df.groupBy("month_num", "arrival_date_month")
                .agg(F.countDistinct("arrival_date_year").alias("years_of_data"),
                     F.sum(1 - F.col("is_canceled")).alias("arrivals"),
                     F.round(F.avg("adr"), 2).alias("avg_adr"),
                     F.round(100 * F.avg("is_canceled"), 1).alias("cancel_rate_pct"),
                     F.sum("realized_revenue").alias("realized_revenue"))
                .withColumn("arrivals_per_year", F.round(F.col("arrivals") / F.col("years_of_data"), 0))
                .withColumn("revenue_per_year", F.round(F.col("realized_revenue") / F.col("years_of_data"), 0))
                .orderBy("month_num")
                .toPandas())
display(monthly_df)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=monthly_df["arrival_date_month"], y=monthly_df["arrivals_per_year"],
                     name="Arrivals per year"), secondary_y=False)
fig.add_trace(go.Scatter(x=monthly_df["arrival_date_month"], y=monthly_df["avg_adr"],
                         name="Average ADR ($)", mode="lines+markers"), secondary_y=True)
fig.update_layout(title="Demand vs. price by month")
fig.update_yaxes(title_text="Arrivals per year", secondary_y=False)
fig.update_yaxes(title_text="Average ADR ($)", secondary_y=True)
fig.show()

month_num,arrival_date_month,years_of_data,arrivals,avg_adr,cancel_rate_pct,realized_revenue,arrivals_per_year,revenue_per_year
1,January,2,25788,71.13,29.4,4960924.059999999,12894.0,2480462.0
2,February,2,33148,74.28,32.1,7233717.260000005,16574.0,3616859.0
3,March,2,41468,81.52,31.8,1.0344796869999995E7,20734.0,5172398.0
4,April,2,42094,100.48,39.7,1.3477930580000015E7,21047.0,6738965.0
5,May,2,44210,109.09,39.1,1.4926365750000019E7,22105.0,7463183.0
6,June,2,40353,117.3,40.2,1.6329162550000012E7,20177.0,8164581.0
7,July,3,49147,127.83,37.1,2.5692272680000015E7,16382.0,8564091.0
8,August,3,53460,141.35,37.7,3.0415589950000014E7,17820.0,1.013853E7
9,September,2,40051,105.54,38.7,1.4875740509999989E7,20026.0,7437870.0
10,October,2,42214,88.84,38.0,1.1498891410000013E7,21107.0,5749446.0


<!doctype html>

### 8.3 Channel and segment profitability
A segment with a high price but a high cancellation rate can be worth less than it looks. Bubbles in the top-left of the chart are the best: high price and low cancellations.

In [0]:
fig = px.scatter(segment_df, x="cancel_rate_pct", y="avg_adr", size="bookings",
                 color="market_segment", text="market_segment",
                 title="Market segments: price vs. cancellation risk (bubble size = bookings)")
fig.update_traces(textposition="top center")
fig.update_layout(xaxis_title="Cancellation rate (%)", yaxis_title="Average ADR ($)", showlegend=False)
fig.show()

display(segment_df.sort_values("revenue_per_booking", ascending=False))
display(seg_summary(df, "distribution_channel").orderBy(F.desc("revenue_per_booking")))

<!doctype html>

market_segment,bookings,cancel_rate_pct,avg_adr,realized_revenue,lost_revenue,revenue_per_booking
Direct,78969,15.5,116.28,2.5583872E7,6223084.0,323.97
Aviation,1502,19.3,101.17,436429.0,89781.0,290.57
Online TA,354953,37.1,117.83,8.6308427E7,6.5015169E7,243.15
Offline TA/TO,146993,33.1,87.43,3.5462892E7,1.4746493E7,241.26
Corporate,32883,18.0,69.57,3624351.0,1167934.0,110.22
Groups,116301,59.4,79.28,1.151261E7,1.5993232E7,98.99
Complementary,4650,13.6,3.05,30039.0,1126.0,6.46
Undefined,21,100.0,17.14,0.0,684.0,0.0


distribution_channel,bookings,cancel_rate_pct,avg_adr,realized_revenue,lost_revenue,revenue_per_booking
Direct,91300,17.5,107.53,2.695447E7,7134200.0,295.23
TA/TO,602332,40.4,104.07,1.30595759E8,9.4293229E7,216.82
GDS,1209,19.1,122.35,217260.0,69782.0,179.7
Corporate,41387,21.2,69.58,5188314.0,1730480.0,125.36
Undefined,44,88.6,50.82,2818.0,9812.0,64.05


### 8.4 Upselling: room types, special requests, meals and parking

In [0]:
room_df = seg_summary(df, "room_changed").toPandas()
print(f"Bookings that got a different room type than reserved: {100 * df.agg(F.avg('room_changed')).first()[0]:.1f}%")
display(room_df)

requests_df = seg_summary(df, "total_of_special_requests").orderBy("total_of_special_requests").toPandas()
fig = px.bar(requests_df, x="total_of_special_requests", y="cancel_rate_pct", text="cancel_rate_pct",
             title="Cancellation rate by number of special requests")
fig.update_layout(xaxis_title="Special requests", yaxis_title="Cancellation rate (%)")
fig.show()

display(seg_summary(df, "meal").orderBy(F.desc("bookings")))
display(seg_summary(df, "required_car_parking_spaces").orderBy("required_car_parking_spaces"))

Bookings that got a different room type than reserved: 12.5%


room_changed,bookings,cancel_rate_pct,avg_adr,realized_revenue,lost_revenue,revenue_per_booking
1,91919,5.4,85.74,2.0851903E7,1205877.0,226.85
0,644353,40.9,104.99,1.42106718E8,1.02031626E8,220.54


<!doctype html>

meal,bookings,cancel_rate_pct,avg_adr,realized_revenue,lost_revenue,revenue_per_booking
BB,569335,36.6,100.1,1.17378014E8,7.6897663E7,206.17
HB,89030,34.7,121.26,3.1097954E7,1.683426E7,349.3
SC,66255,37.1,99.41,1.067742E7,7845158.0,161.16
Undefined,6852,22.7,92.32,2308250.0,612940.0,336.87
FB,4800,58.5,109.89,1496982.0,1047481.0,311.87


required_car_parking_spaces,bookings,cancel_rate_pct,avg_adr,realized_revenue,lost_revenue,revenue_per_booking
0,688990,39.0,101.89,1.44892802E8,1.03237503E8,210.3
1,47088,0.0,112.8,1.7972949E7,0.0,381.69
2,168,0.0,118.88,74007.0,0.0,440.52
3,19,0.0,130.26,17223.0,0.0,906.47
8,7,0.0,51.43,1640.0,0.0,234.29


**Caution:** room types are usually assigned at check-in, and canceled bookings never check in. Room changes will therefore look strongly linked to low cancellation even though the change doesn't cause it. Use this table to see how often upgrades happen (an upsell opportunity), not as a cancellation driver.

### 8.5 Length of stay by customer type (stays that happened)

In [0]:
stay_df = (df.filter(F.col("is_canceled") == 0)
             .groupBy("customer_type")
             .agg(F.count("*").alias("stays"),
                  F.round(F.avg("total_nights"), 2).alias("avg_nights"),
                  F.round(100 * F.avg(F.col("stays_in_weekend_nights") / F.col("total_nights")), 1).alias("weekend_night_share_pct"),
                  F.round(F.avg("adr"), 2).alias("avg_adr"),
                  F.round(F.avg("revenue"), 2).alias("revenue_per_stay"))
             .orderBy(F.desc("revenue_per_stay"))
             .toPandas())
display(stay_df)

fig = px.bar(stay_df, x="customer_type", y="revenue_per_stay", text_auto=".0f",
             hover_data=["avg_nights", "avg_adr", "stays"],
             title="Average revenue per stay by customer type")
fig.update_layout(xaxis_title=None, yaxis_title="Revenue per stay ($)")
fig.show()

customer_type,stays,avg_nights,weekend_night_share_pct,avg_adr,revenue_per_stay
Contract,17688,6.26,26.4,92.46,536.28
Transient,331209,3.38,26.7,105.89,366.6
Transient-Party,115683,3.08,24.0,88.68,270.97
Group,3253,2.71,30.9,80.7,217.14


<!doctype html>

### 8.6 Repeat guests

In [0]:
repeat_df = seg_summary(df, "is_repeated_guest").orderBy("is_repeated_guest").toPandas()
repeat_df["share_of_bookings_pct"] = (100 * repeat_df["bookings"] / repeat_df["bookings"].sum()).round(1)
display(repeat_df)

is_repeated_guest,bookings,cancel_rate_pct,avg_adr,realized_revenue,lost_revenue,revenue_per_booking,share_of_bookings_pct
0,714472,37.1,103.58,1.59884808E8,1.02667133E8,223.78,97.0
1,21800,14.8,70.1,3073813.0,570370.0,141.0,3.0


### 8.7 Countries: high value and reliable
The guest map in section 5 shows volume. This shows which countries bring revenue **and** follow through on their bookings. Only countries with at least `MIN_COUNTRY_BOOKINGS` bookings are included, so small countries don't produce misleading rates.

In [0]:
MIN_COUNTRY_BOOKINGS = 1000

country_df = (seg_summary(df, "country", min_bookings=MIN_COUNTRY_BOOKINGS)
                .orderBy(F.desc("realized_revenue")).limit(15).toPandas())
display(country_df)

fig = px.scatter(country_df, x="realized_revenue", y="cancel_rate_pct", size="bookings", text="country",
                 title="Top 15 countries: revenue vs. cancellation rate")
fig.update_traces(textposition="top center")
fig.update_layout(xaxis_title="Realized revenue ($)", yaxis_title="Cancellation rate (%)")
fig.show()

country,bookings,cancel_rate_pct,avg_adr,realized_revenue,lost_revenue,revenue_per_booking
PRT,290837,55.9,92.97,3.4360827E7,5.1440188E7,118.14
GBR,76338,19.9,95.95,2.6076412E7,7363524.0,341.59
FRA,65880,18.7,110.0,1.9562272E7,5963666.0,296.94
ESP,53387,25.7,118.4,1.4054506E7,6732932.0,263.26
DEU,44983,16.4,104.03,1.2744174E7,3416865.0,283.31
IRL,21061,24.8,97.74,7721102.0,2535946.0,366.61
ITA,23726,35.4,113.79,5462147.0,3473086.0,230.22
BEL,14924,20.3,114.37,4905832.0,1393251.0,328.72
NLD,13129,18.2,108.99,3953607.0,1042394.0,301.14
CHE,11172,25.3,123.23,3560016.0,1637093.0,318.66


<!doctype html>

### 8.8 Lead time vs. price: do early bookers pay more or less?

In [0]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=lead_df["lead_time_bucket"].astype(str), y=lead_df["avg_adr"], name="Average ADR ($)"),
              secondary_y=False)
fig.add_trace(go.Scatter(x=lead_df["lead_time_bucket"].astype(str), y=lead_df["cancel_rate_pct"],
                         name="Cancellation rate (%)", mode="lines+markers"), secondary_y=True)
fig.update_layout(title="Price and cancellation rate by lead time")
fig.update_yaxes(title_text="Average ADR ($)", secondary_y=False)
fig.update_yaxes(title_text="Cancellation rate (%)", secondary_y=True)
fig.show()

<!doctype html>

## 9. Key insights
Generated from the tables above, so the numbers always match the data.

In [0]:
booked_total = kpi["realized"] + kpi["lost"]
seg_top = segment_df.iloc[0]
lead_hi, lead_lo = lead_df.loc[lead_df["cancel_rate_pct"].idxmax()], lead_df.loc[lead_df["cancel_rate_pct"].idxmin()]
adr_hi, adr_lo = monthly_df.loc[monthly_df["avg_adr"].idxmax()], monthly_df.loc[monthly_df["avg_adr"].idxmin()]
busy = monthly_df.loc[monthly_df["arrivals_per_year"].idxmax()]
busy_raw = monthly_df.loc[monthly_df["arrivals"].idxmax()]
best_seg = segment_df[segment_df["bookings"] >= 0.01 * kpi["bookings"]].sort_values("revenue_per_booking").iloc[-1]
new, rep = repeat_df.set_index("is_repeated_guest").loc[0], repeat_df.set_index("is_repeated_guest").loc[1]
top_country = country_df.iloc[0]
no_dep = deposit_df.set_index("deposit_type")

insights = [
    f"{100 * kpi['cancel_rate']:.1f}% of bookings are canceled, costing ${kpi['lost']:,.0f} "
    f"({100 * kpi['lost'] / booked_total:.1f}% of all booked room revenue).",
    f"Non Refund bookings make up ${no_dep.loc['Non Refund', 'lost_revenue']:,.0f} of that, but the hotel keeps those deposits, "
    f"so the real loss is about ${kpi['lost'] - no_dep.loc['Non Refund', 'lost_revenue']:,.0f}.",
    f"{seg_top['market_segment']} accounts for {100 * seg_top['lost_revenue'] / kpi['lost']:.0f}% of lost revenue "
    f"(cancellation rate {seg_top['cancel_rate_pct']}%).",
    f"Cancellation rate is lowest for bookings made {lead_lo['lead_time_bucket']} ahead ({lead_lo['cancel_rate_pct']}%) "
    f"and highest for {lead_hi['lead_time_bucket']} ahead ({lead_hi['cancel_rate_pct']}%).",
    "Cancellation rate by deposit type: " + ", ".join(f"{k} {v}%" for k, v in no_dep["cancel_rate_pct"].items()) + ".",
    f"Prices peak in {adr_hi['arrival_date_month']} (avg ADR ${adr_hi['avg_adr']:.0f}) and bottom out in "
    f"{adr_lo['arrival_date_month']} (${adr_lo['avg_adr']:.0f}). After adjusting for partial years, the busiest "
    f"month is {busy['arrival_date_month']}" + (f" (raw totals would wrongly say {busy_raw['arrival_date_month']})."
                                                 if busy_raw['arrival_date_month'] != busy['arrival_date_month'] else "."),
    f"{best_seg['market_segment']} brings the most revenue per booking made (${best_seg['revenue_per_booking']:.0f}).",
    f"Repeat guests are {repeat_df.loc[repeat_df['is_repeated_guest'] == 1, 'share_of_bookings_pct'].iloc[0]}% of bookings "
    f"and cancel {rep['cancel_rate_pct']}% of the time vs {new['cancel_rate_pct']}% for new guests.",
    f"{top_country['country']} is the top revenue country (${top_country['realized_revenue']:,.0f}) "
    f"with a {top_country['cancel_rate_pct']}% cancellation rate.",
]
for i, s in enumerate(insights, 1):
    print(f"{i}. {s}")

1. 36.5% of bookings are canceled, costing $103,237,503 (38.8% of all booked room revenue).
2. Non Refund bookings make up $20,294,830 of that, but the hotel keeps those deposits, so the real loss is about $82,942,673.
3. Online TA accounts for 63% of lost revenue (cancellation rate 37.1%).
4. Cancellation rate is lowest for bookings made 0-7 days ahead (9.8%) and highest for 365+ days ahead (67.4%).
5. Cancellation rate by deposit type: Non Refund 99.2%, No Deposit 28.6%, Refundable 21.8%.
6. Prices peak in August (avg ADR $141) and bottom out in January ($71). After adjusting for partial years, the busiest month is May (raw totals would wrongly say August).
7. Direct brings the most revenue per booking made ($324).
8. Repeat guests are 3.0% of bookings and cancel 14.8% of the time vs 37.1% for new guests.
9. PRT is the top revenue country ($34,360,827) with a 55.9% cancellation rate.


## 10. Recommendations
Each recommendation is tied to the analysis section and insight that supports it.

1. **Ask for a deposit on long-lead bookings (8.1, insights 2, 4 and 5).** Cancellation rate climbs steadily with lead time, and almost all of the real loss comes from "No Deposit" bookings. Requiring a partial prepayment for bookings made 90+ days ahead, plus a reconfirmation email about 30 days before arrival, targets the riskiest bookings without affecting last-minute guests.
2. **Tighten terms for the riskiest segments (8.1, 8.3, insight 3).** Online TA causes most of the lost revenue by volume, and Groups have the highest cancellation rate. Release unconfirmed group blocks earlier and negotiate stricter cancellation windows with online travel agencies.
3. **Overbook carefully using cancellation patterns (8.1).** Cancellation rates are predictable by segment and lead time, so the hotel can safely accept a few more bookings than rooms on high-demand dates.
4. **Price by season (8.2, insight 6).** ADR peaks in July–August. Months with high arrivals but a lower ADR than the peak have room for a rate increase. November–February are the slow months, so use promotions there.
5. **Grow direct bookings (8.3, insight 7).** Direct bookings bring the most revenue per booking made and cancel much less than travel-agent bookings. Offer a best-rate guarantee or small perks for booking on the hotel's own website.
6. **Start a loyalty program (8.6, insight 8).** Repeat guests are a small share of bookings but cancel far less often. Keeping them is cheaper than winning new guests.
7. **Sell upgrades and bundles (8.4, 8.5).** About 1 in 8 guests already end up in a different room type, which could be a paid upgrade at check-in. Guests on half board and guests who reserve parking have a higher revenue per booking, so offer meal and parking bundles at booking time.
8. **Investigate Non Refund bookings with the data owner (8.1, insight 5).** A cancellation rate near 100% for non-refundable deposits is unusual. It may reflect how the booking system records these deposits. Flag it as a data question before using it to make policy decisions.

## 12. Export the clean table for Power BI
This saves the cleaned data, with all the new revenue columns, as a table. In Power BI Desktop: **Get Data → Azure Databricks** (or **Databricks**), and enter the *Server hostname* and *HTTP path* from **SQL Warehouses → your warehouse → Connection details** in Databricks.

In [0]:
(df.write.mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable("workspace.default.hotel_bookings_clean"))

print("Saved workspace.default.hotel_bookings_clean:", spark.table("workspace.default.hotel_bookings_clean").count(), "rows")

Saved workspace.default.hotel_bookings_clean: 736272 rows
